In [7]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/student_data.csv")

# Basic inspection
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nFirst 5 rows:")
print(df.head())

Shape: (25000, 10)

Columns:
['Student ID', 'Academic Major', 'GPA', 'Skill', 'Location Interest', 'University', 'School Year', 'Area of Experience', 'Number of Experience (yrs)', 'Degree']

Data types:
Student ID                     object
Academic Major                 object
GPA                           float64
Skill                          object
Location Interest              object
University                     object
School Year                    object
Area of Experience             object
Number of Experience (yrs)    float64
Degree                         object
dtype: object

Missing values:
Student ID                    0
Academic Major                0
GPA                           0
Skill                         0
Location Interest             0
University                    0
School Year                   0
Area of Experience            0
Number of Experience (yrs)    0
Degree                        0
dtype: int64

First 5 rows:
  Student ID          Academic Major  

#  Structural Validity Test

We want to prove that the dataset does not violate hard logical rules.


## Degree vs School Year

Expected mappings:

Bachelor's → Freshman, Sophomore, Junior, Senior
Master's → 1st Year Master's, 2nd Year Master's
PhD → 1st–5th Year PhD

## GPA range

A realistic GPA should satisfy:

- 0≤ GPA ≤ 4.0

## Experience must be nonnegative

Experience ≥ 0

In [8]:
# Structural validity rules


valid_years_by_degree = {
    "Bachelor's": {"Freshman", "Sophomore", "Junior", "Senior"},
    "Master's": {"1st Year Master's", "2nd Year Master's"},
    "PhD": {
        "1st Year PhD", "2nd Year PhD", "3rd Year PhD",
        "4th Year PhD", "5th Year PhD"
    }
}

# Rule 1: valid degree-school year pairing
df["invalid_degree_year"] = ~df.apply(
    lambda row: row["School Year"] in valid_years_by_degree.get(row["Degree"], set()),
    axis=1
)

# Rule 2: GPA outside bounds
df["invalid_gpa"] = ~df["GPA"].between(0.0, 4.0)

# Rule 3: negative experience
df["invalid_experience"] = df["Number of Experience (yrs)"] < 0

# Combine contradictions
rule_cols = ["invalid_degree_year", "invalid_gpa", "invalid_experience"]
df["any_structural_violation"] = df[rule_cols].any(axis=1)

# Summary
n = len(df)
violation_summary = pd.DataFrame({
    "Rule": rule_cols + ["any_structural_violation"],
    "Violations": [
        df["invalid_degree_year"].sum(),
        df["invalid_gpa"].sum(),
        df["invalid_experience"].sum(),
        df["any_structural_violation"].sum()
    ]
})

violation_summary["Violation Rate"] = violation_summary["Violations"] / n

print(violation_summary)

                       Rule  Violations  Violation Rate
0       invalid_degree_year           0             0.0
1               invalid_gpa           0             0.0
2        invalid_experience           0             0.0
3  any_structural_violation           0             0.0


The synthetic dataset satisfied all deterministic structural constraints, with an observed contradiction rate of 0%.

In [9]:
# 95% CI for contradiction rate

p_hat = df["any_structural_violation"].mean()
n = len(df)

se = np.sqrt((p_hat * (1 - p_hat)) / n)
lower = p_hat - 1.96 * se
upper = p_hat + 1.96 * se

# Clip to [0,1]
lower = max(0, lower)
upper = min(1, upper)

print(f"Contradiction rate: {p_hat:.6f}")
print(f"95% CI: [{lower:.6f}, {upper:.6f}]")

Contradiction rate: 0.000000
95% CI: [0.000000, 0.000000]


If the rate is zero, the normal CI formula becomes degenerate. No structural contradictions were observed in 25000 rows.

## Monotonic progression test for academic realism

A realistic student dataset should reflect academic progression:
- earlier years should have less experience
- later years should have more experience

Freshman < Sophomore < Junior < Senior < Master′s1 < Master′s2 < PhD1 <⋯< PhD5

Then we test whether Number of Experience (yrs) increases with this ordering.

Why Spearman? Because school year is ordinal, not continuous. Spearman tests whether experience rises monotonically with school year.

In [10]:
from scipy.stats import spearmanr

# School year ordering

year_order = {
    "Freshman": 1,
    "Sophomore": 2,
    "Junior": 3,
    "Senior": 4,
    "1st Year Master's": 5,
    "2nd Year Master's": 6,
    "1st Year PhD": 7,
    "2nd Year PhD": 8,
    "3rd Year PhD": 9,
    "4th Year PhD": 10,
    "5th Year PhD": 11
}

df["school_year_ord"] = df["School Year"].map(year_order)

# Spearman correlation
rho, p_value = spearmanr(df["school_year_ord"], df["Number of Experience (yrs)"])

print(f"Spearman rho: {rho:.4f}")
print(f"p-value: {p_value:.6g}")

Spearman rho: 0.8807
p-value: 0


In [11]:
grouped_exp = (
    df.groupby("School Year")["Number of Experience (yrs)"]
      .agg(["count", "mean", "std"])
      .reset_index()
)

grouped_exp["order"] = grouped_exp["School Year"].map(year_order)
grouped_exp = grouped_exp.sort_values("order").drop(columns="order")

print(grouped_exp)

          School Year  count      mean       std
7            Freshman   4314  0.504474  0.260261
10          Sophomore   4485  1.149342  0.376421
8              Junior   5087  1.902162  0.552116
9              Senior   5399  2.544971  0.763817
0   1st Year Master's   2474  2.786985  0.699639
2   2nd Year Master's   2004  3.823004  0.890402
1        1st Year PhD    220  3.662727  0.921222
3        2nd Year PhD    268  4.462687  0.990152
4        3rd Year PhD    290  5.305862  1.068388
5        4th Year PhD    253  6.169565  1.189063
6        5th Year PhD    206  7.149515  1.462332


Mean years of experience increase strongly with academic progression, with only a minor local deviation between 2nd-year Master’s and 1st-year PhD students

## Test whether Degree and School Year are strongly associated

In real academic data, degree level and school year are not independent.
Using the chi-square test of independence then we report Cramér’s V as an effect size.

In [12]:
from scipy.stats import chi2_contingency

# Degree vs School Year

contingency = pd.crosstab(df["Degree"], df["School Year"])
chi2, p, dof, expected = chi2_contingency(contingency)

n = contingency.to_numpy().sum()
r, c = contingency.shape
cramers_v = np.sqrt(chi2 / (n * min(r - 1, c - 1)))

print("Chi-square statistic:", chi2)
print("p-value:", p)
print("Degrees of freedom:", dof)
print("Cramer's V:", cramers_v)

print("\nContingency table:")
print(contingency)

Chi-square statistic: 50000.00000000001
p-value: 0.0
Degrees of freedom: 20
Cramer's V: 1.0

Contingency table:
School Year  1st Year Master's  1st Year PhD  2nd Year Master's  2nd Year PhD  \
Degree                                                                          
Bachelor's                   0             0                  0             0   
Master's                  2474             0               2004             0   
PhD                          0           220                  0           268   

School Year  3rd Year PhD  4th Year PhD  5th Year PhD  Freshman  Junior  \
Degree                                                                    
Bachelor's              0             0             0      4314    5087   
Master's                0             0             0         0       0   
PhD                   290           253           206         0       0   

School Year  Senior  Sophomore  
Degree                          
Bachelor's     5399       4485  
Master'

# Major vs Area of Experience association

In [ ]:
# Academic Major vs Area of Experience
# A strong association would suggest that certain majors are more likely to have experience in specific areas, which could be expected. A weak association might indicate that experience areas are more evenly distributed across majors.

contingency_major_area = pd.crosstab(df["Academic Major"], df["Area of Experience"])
chi2_ma, p_ma, dof_ma, expected_ma = chi2_contingency(contingency_major_area)

n_ma = contingency_major_area.to_numpy().sum()
r_ma, c_ma = contingency_major_area.shape
cramers_v_ma = np.sqrt(chi2_ma / (n_ma * min(r_ma - 1, c_ma - 1)))

print("Chi-square statistic:", chi2_ma)
print("p-value:", p_ma)
print("Degrees of freedom:", dof_ma)
print("Cramer's V:", cramers_v_ma)

Chi-square statistic: 265151.5748709447
p-value: 0.0
Degrees of freedom: 944
Cramer's V: 0.8141737757858342
